# PhysDual-GNN — Fase 2: Baseline GraphDTA (Treinamento Oficial e Avaliação)

Este notebook executa o treinamento completo, benchmarking de tempo, persistência por run e validação dos baselines GraphDTA no Google Colab (GPU L4 / T4).

### Diretrizes Arquiteturais Atendidas:
- **C1 (Isolamento Estrito de Teste):** O conjunto de teste é avaliado unicamente após o encerramento do treinamento com restauração do `checkpoint_best.pt`.
- **C2 (Persistência Individual e Retomabilidade):** Cada combinação `(model, dataset, split, seed)` salva um JSON atômico e suporta retomada a partir de `checkpoint_last.pt`.
- **C3 (Idempotência Estrita):** Runs finalizadas são identificadas e puladas sem reprocessamento redundante.
- **Auditoria de Gates:** Validação via `scripts/verify_phase2.py` ao término do pipeline.

In [ ]:
# 1. Setup de Ambiente, Verificação de GPU e Montagem do Google Drive
import os
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    drive_root = Path("/content/drive/MyDrive/physdual-gnn")
    if drive_root.exists():
        os.environ.setdefault("PHYSDUAL_ROOT", str(drive_root))
        os.chdir(str(drive_root))
    print(f"Diretório de trabalho: {os.getcwd()}")

repo_root = Path(os.environ.get("PHYSDUAL_ROOT", Path.cwd())).resolve()
src_path = repo_root / "src"
scripts_path = repo_root / "scripts"
for p in (src_path, scripts_path):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

!nvidia-smi

In [ ]:
# 2. Instalação de Dependências Complementares (se necessário)
try:
    import lifelines
    import rdkit
    import torch_geometric
    print("Dependências de Fase 2 (lifelines, rdkit, torch_geometric) já instaladas!")
except ImportError:
    print("Instalando dependências ausentes...")
    !pip install -q lifelines rdkit torch-geometric

In [ ]:
# 3. Verificação de Pré-Requisitos: Gate da Fase 1 (Dados, Splits e Embeddings)
!python scripts/verify_phase1.py

In [ ]:
# 4. Passo 1: Benchmark de Tempo de GPU (3 épocas por dataset)
# Mede sec_per_epoch real e projeta o tempo para 500 épocas em Davis e KIBA
!python scripts/run_phase2_baseline.py --benchmark-epochs 3

import json
bench_file = repo_root / "outputs" / "results" / "phase2_timing_benchmark.json"
if bench_file.exists():
    print("\n--- Resumo do Benchmark de Tempo ---")
    print(json.dumps(json.loads(bench_file.read_text(encoding="utf-8")), indent=2))

In [ ]:
# 5. Passo 2: Treinamento Oficial Davis — GraphDTA GIN (3 splits x 5 seeds)
# Executa random, cold_drug e cold_target com paciência adaptativa e persistência atômica
!python scripts/run_phase2_baseline.py \
    --model graphdta_gin \
    --dataset davis \
    --split all \
    --seeds 0,1,2,3,4 \
    --epochs 500 \
    --batch-size 512 \
    --lr 5e-4

In [ ]:
# 6. Passo 3: Treinamento Oficial KIBA — GraphDTA GIN (3 splits x 5 seeds)
!python scripts/run_phase2_baseline.py \
    --model graphdta_gin \
    --dataset kiba \
    --split all \
    --seeds 0,1,2,3,4 \
    --epochs 500 \
    --batch-size 512 \
    --lr 5e-4

In [ ]:
# 7. Passo 4: Variante Secundária Davis — GraphDTA CNN3 (3 splits x 5 seeds)
!python scripts/run_phase2_baseline.py \
    --model graphdta_cnn3 \
    --dataset davis \
    --split all \
    --seeds 0,1,2,3,4 \
    --epochs 500 \
    --batch-size 512 \
    --lr 5e-4

In [ ]:
# 8. Passo 5 (Condicional): Treinamento Variante Secundária KIBA — GraphDTA CNN3
# Avalia a elegibilidade apurada no benchmark (sec_per_epoch <= 15s)
eligible = True
if bench_file.exists():
    try:
        bdata = json.loads(bench_file.read_text(encoding="utf-8"))
        eligible = bdata.get("datasets", {}).get("kiba", {}).get("eligible_for_cnn3_secondary", True)
    except Exception:
        pass

if eligible:
    print("Benchmark autorizou execução da variante secundária no KIBA. Iniciando...")
    !python scripts/run_phase2_baseline.py \
        --model graphdta_cnn3 \
        --dataset kiba \
        --split all \
        --seeds 0,1,2,3,4 \
        --epochs 500 \
        --batch-size 512 \
        --lr 5e-4
else:
    print("KIBA cnn3 omitido por restrição de orçamento computacional apurada no benchmark (> 15s/época).")

In [ ]:
# 9. Passo 6: Agregação Estatística e Verificação Oficial do Gate de Pronto da Fase 2
!python scripts/run_phase2_baseline.py --aggregate-only
!python scripts/verify_phase2.py --full

In [ ]:
# 10. Passo 7: Exibição da Tabela Comparativa de Resultados Finais
import json
from pathlib import Path
import pandas as pd

results_dir = repo_root / "outputs" / "results"
records = []

for agg_path in sorted(results_dir.glob("baseline_*.json")):
    data = json.loads(agg_path.read_text(encoding="utf-8"))
    m = data.get("model", "")
    ds = data.get("dataset", "")
    sp = data.get("split", "")
    agg = data.get("aggregated", {})
    t_metrics = agg.get("test_metrics_full", {})
    triv = agg.get("trivial_baselines", {})

    def fmt(m_dict, key):
        mean = m_dict.get(key, {}).get("mean")
        std = m_dict.get(key, {}).get("std")
        if mean is None or pd.isna(mean):
            return "N/A"
        return f"{mean:.4f} ± {std:.4f}" if std is not None else f"{mean:.4f}"

    records.append({
        "Modelo": m,
        "Dataset": ds,
        "Split": sp,
        "CI (Test)": fmt(t_metrics, "ci"),
        "MSE (Test)": fmt(t_metrics, "mse"),
        "Pearson": fmt(t_metrics, "pearson"),
        "Spearman": fmt(t_metrics, "spearman"),
        "Mean Baseline MSE": fmt(triv.get("mean_predictor", {}), "mse"),
        "Ridge Baseline MSE": fmt(triv.get("ridge_aa_composition", {}), "mse"),
        "Best Epoch": fmt(agg, "best_epoch"),
        "Sec/Epoch": fmt(agg, "sec_per_epoch"),
    })

if records:
    df_summary = pd.DataFrame(records)
    display(df_summary)
    print("\n=== Markdown Table ===\n")
    print(df_summary.to_markdown(index=False))
else:
    print("Nenhum arquivo baseline consolidado encontrado em:", results_dir)

## 11. Protocolo v2 (Opção C): GroupKFold de 5 Dobras no Davis e Extensão KIBA

Nesta seção implementamos as diretrizes do **ADR #13** acordadas na auditoria forense:
1. **5-Fold GroupKFold no Davis** (`cold_drug` e `cold_target`), agrupado estritamente por conteúdo (`smiles` canônico / `sequence`), com 100% de cobertura do teste sem sobreposição entre dobras e validação interna (~10% grupos).
2. **Treinamento dos 2 Modelos** (`graphdta_gin` oficial e `graphdta_cnn3` estilo DeepDTA) exatamente nas mesmas dobras.
3. **Persistência de Previsões Fora da Dobra (OOF)** em CSV para análise de resíduos e cálculo de métricas globais.
4. **Métricas de Classificação Binária** (AUROC e AUPR nos limiares $pK_d > 5.0$ e $pK_d \ge 7.0$).
5. **Bootstrap Agrupado (Cluster-Level)** com IC de 95% para $\Delta\text{MSE}$ e $\Delta\text{CI}$ contra baselines marginais de promiscuidade (`drug_mean_predictor` e `target_mean_predictor`).
6. **Controle de Épocas no KIBA**: Extensão das seeds 1 e 3 de 500 para 1000 épocas com restauração atômica do estado de early stopping.

In [ ]:
# Passo 11.1: Geração dos Splits GroupKFold (5 Dobras) para Davis
import pandas as pd
from pathlib import Path
from physdual.data.splits import create_group_kfold_splits, save_splits, verify_group_kfold_coverage

davis_csv = Path('data/processed/davis_clean.csv')
if not davis_csv.exists():
    !python scripts/run_phase1_data.py --skip-esm --dataset davis

df_davis = pd.read_csv(davis_csv)
for split_type, group_col in [('cold_drug', 'smiles'), ('cold_target', 'sequence')]:
    kfold_splits = create_group_kfold_splits(
        df_davis, group_col=group_col, n_splits=5, val_ratio=0.10, random_state=42
    )
    verify_group_kfold_coverage(df_davis, kfold_splits, group_col=group_col)
    for fold_id, s_dict in enumerate(kfold_splits):
        out_path = Path(f'data/processed/splits/davis_{split_type}_fold{fold_id}.json')
        save_splits(s_dict, out_path)
    print(f'[OK] 5 dobras GroupKFold geradas e validadas para davis_{split_type}')


In [ ]:
# Passo 11.2: Treinamento Davis GroupKFold (5 Dobras) — GraphDTA GIN e CNN3
# Executa os dois modelos nas 5 dobras para cold_drug e cold_target

# 1. GraphDTA GIN — cold_drug e cold_target
!python scripts/run_phase2_baseline.py \
    --protocol kfold \
    --model graphdta_gin \
    --dataset davis \
    --split cold_drug \
    --folds 0,1,2,3,4 \
    --epochs 500 \
    --davis-cold-patience 80 \
    --batch-size 512 \
    --lr 5e-4

!python scripts/run_phase2_baseline.py \
    --protocol kfold \
    --model graphdta_gin \
    --dataset davis \
    --split cold_target \
    --folds 0,1,2,3,4 \
    --epochs 500 \
    --davis-cold-patience 80 \
    --batch-size 512 \
    --lr 5e-4

# 2. GraphDTA CNN3 — cold_drug e cold_target
!python scripts/run_phase2_baseline.py \
    --protocol kfold \
    --model graphdta_cnn3 \
    --dataset davis \
    --split cold_drug \
    --folds 0,1,2,3,4 \
    --epochs 500 \
    --davis-cold-patience 80 \
    --batch-size 512 \
    --lr 5e-4

!python scripts/run_phase2_baseline.py \
    --protocol kfold \
    --model graphdta_cnn3 \
    --dataset davis \
    --split cold_target \
    --folds 0,1,2,3,4 \
    --epochs 500 \
    --davis-cold-patience 80 \
    --batch-size 512 \
    --lr 5e-4


In [ ]:
# Passo 11.3: Controle de Épocas KIBA random (Extensão Seeds 1 e 3 de 500 para 1000 épocas)
# Retoma o otimizador e o contador de early stopping preservado no checkpoint_last.pt
!python scripts/run_phase2_baseline.py \
    --model graphdta_gin \
    --dataset kiba \
    --split random \
    --seeds 1,3 \
    --epochs 1000 \
    --extend-epochs \
    --batch-size 512 \
    --lr 5e-4


In [ ]:
# Passo 11.4: Exibição Consolidada dos Resultados K-Fold, OOF e Bootstrap Agrupado
import glob
import json
from pathlib import Path

kfold_files = sorted(glob.glob('outputs/results/baseline_*_kfold.json'))
print(f'Encontrados {len(kfold_files)} arquivos de resultados K-Fold.')

for kf in kfold_files:
    data = json.loads(Path(kf).read_text(encoding='utf-8'))
    print('\n' + '=' * 75)
    print(f"Modelo: {data['model']} | Dataset: {data['dataset']} | Split: {data['split']} (5-Fold GroupKFold)")
    print('=' * 75)
    
    print('Métricas Médias entre Dobras (Média ± Desvio Amostral):')
    for m in ['ci', 'mse', 'rmse', 'pearson', 'spearman', 'rm2']:
        s = data.get('per_fold_mean_std', {}).get(m, {})
        print(f"  {m.upper():<10}: {s.get('mean', 0.0):.4f} ± {s.get('std', 0.0):.4f}")
    
    print('\nMétricas Globais Fora da Dobra (OOF Pooled):')
    for m in ['ci', 'mse', 'rmse', 'pearson', 'spearman', 'rm2']:
        v = data.get('global_oof_metrics', {}).get(m, 0.0)
        print(f"  {m.upper():<10}: {v:.4f}")
        
    print('\nClassificação Binária:')
    for th, bm in data.get('binary_classification_metrics', {}).items():
        print(f"  {th}: AUROC = {bm.get('auroc', 0.0):.4f} | AUPR = {bm.get('aupr', 0.0):.4f}")
        
    print('\nBootstrap Agrupado no Nível de Cluster (IC 95%):')
    for bl_name in ['vs_drug_mean', 'vs_target_mean']:
        bs = data.get('bootstrap_analysis', {}).get(bl_name)
        if bs and 'delta_mse' in bs:
            dmse = bs['delta_mse']
            dci = bs['delta_ci']
            print(f"  {bl_name}: Delta MSE = {dmse['mean']:+.4f} (95% CI: [{dmse['ci_95'][0]:.4f}, {dmse['ci_95'][1]:.4f}]) | p(Sup) = {bs.get('p_superiority_mse', 0):.3f}")
            print(f"             Delta CI  = {dci['mean']:+.4f} (95% CI: [{dci['ci_95'][0]:.4f}, {dci['ci_95'][1]:.4f}]) | p(Sup) = {bs.get('p_superiority_ci', 0):.3f}")
    print(f"Arquivo OOF CSV: {data.get('oof_csv_path')}")
